# Fine-tuning LoRA pour Stable Diffusion

In [ ]:
import os
import gc
import json
import torch
import torch.nn.functional as F
from torchvision import transforms
from diffusers import UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

gc.collect()
torch.cuda.empty_cache()

c:\VENV\ML\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"
dataset_dir = "./Iwaria_Press_Dataset/images"
output_dir = "./output"
resolution = 512
train_batch_size = 2
num_train_epochs = 10
learning_rate = 1e-4
adam_weight_decay = 1e-2

os.makedirs(output_dir, exist_ok=True)

## 1. Chargement des composants du modèle de base

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")

text_encoder = CLIPTextModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="text_encoder",
    torch_dtype=torch.float16, low_cpu_mem_usage=True
)
text_encoder.requires_grad_(False)
gc.collect()

vae = AutoencoderKL.from_pretrained(
    pretrained_model_name_or_path, subfolder="vae",
    torch_dtype=torch.float16, low_cpu_mem_usage=True
)
vae.requires_grad_(False)
gc.collect()

unet = UNet2DConditionModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="unet",
    torch_dtype=torch.float16, low_cpu_mem_usage=True,
    device_map="auto"
)
unet.requires_grad_(False)
gc.collect()

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 221.03it/s]
CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\VENV\ML\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


## 2. Injection des couches LoRA dans l'UNet

In [ ]:
lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.0,
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()
unet.train()

## 3. Optimiseur

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=learning_rate,
    weight_decay=adam_weight_decay,
)

## 4. Préparation du jeu de données

In [ ]:
# Générer metadata.jsonl conforme à partir du fichier source
src_jsonl = os.path.join(dataset_dir, "dataset_metadata_IACimages.jsonl")
dst_jsonl = os.path.join(dataset_dir, "metadata.jsonl")

with open(src_jsonl, "r", encoding="utf-8") as src, open(dst_jsonl, "w", encoding="utf-8") as dst:
    for line in src:
        entry = json.loads(line)
        dst.write(json.dumps({
            "file_name": entry["Nom_Fichier"],
            "text": entry["Legende_Explicite"]
        }) + "\n")

print(f"metadata.jsonl généré dans {dst_jsonl}")

dataset = load_dataset("imagefolder", data_dir=dataset_dir, split="train")
print("Colonnes disponibles:", dataset.column_names)

train_transforms = transforms.Compose([
    transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(resolution),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def preprocess(examples):
    images = [train_transforms(image.convert("RGB")) for image in examples["image"]]
    inputs = tokenizer(examples["text"], max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt")
    return {"pixel_values": images, "input_ids": inputs.input_ids}

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    input_ids = torch.stack([example["input_ids"] for example in examples])
    return {"pixel_values": pixel_values, "input_ids": input_ids}

train_dataset = dataset.with_transform(preprocess)
train_dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=collate_fn
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entraînement sur : {device}")
vae.to(device)
text_encoder.to(device)
unet.to(device)

## 5. Boucle d'entraînement

In [ ]:
for epoch in range(num_train_epochs):
    progress_bar = tqdm(total=len(train_dataloader), desc=f"Epoch {epoch+1}")
    for step, batch in enumerate(train_dataloader):
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)

        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
            encoder_hidden_states = text_encoder(input_ids)[0].to(dtype=torch.float16)

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, 1000, (latents.shape[0],), device=device).long()
        noisy_latents = latents + noise

        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        progress_bar.update(1)
        progress_bar.set_postfix({"loss": loss.item()})

## 6. Sauvegarde des poids LoRA

In [ ]:
print(f"Sauvegarde du modèle dans {output_dir}...")
unet.save_pretrained(output_dir)
print("Entraînement terminé avec succès !")